In [ ]:
# -*- coding: utf-8 -*-
"""
stage_1_indexing_colpali_qdrant.py
=============================================================================
Stage 1 — Multimodal PDF Image Indexing menggunakan ColPali v1.3 (native
transformers) & Qdrant. Dioptimasi utk Google Colab T4.

PERUBAHAN PENTING DARI VERSI SEBELUMNYA — BACA DULU
-----------------------------------------------------------------------------
Versi sebelumnya pakai library `colpali_engine` (ColPali + checkpoint
"vidore/colpali-v1.3") dan masih GAGAL — LOAD REPORT menunjukkan SEMUA bobot
LoRA adapter (q/k/v/o_proj, mlp gate/up/down_proj, di 18 layer) MISSING
(random-init), padahal LoRA itu JUSTRU inti kemampuan retrieval ColPali.
Tanpanya, model cuma jadi PaliGemma biasa (captioning), bukan model retrieval.

AKAR MASALAH (dikonfirmasi via riset, bukan tebakan):
  `colpali_engine` memuat ColPali sebagai ADAPTER LoRA terpisah di atas base
  model PaliGemma via `peft`. Transformers melakukan refactor struktur
  internal model Gemma/PaliGemma (sekitar v4.50+) yang mengubah penamaan
  layer ("language_model.X" <-> "model.language_model.X" dsb). Adapter LoRA
  lama disimpan dengan penamaan SEBELUM refactor, jadi tidak cocok lagi
  dengan struktur model SESUDAH refactor. Ini bug yang sudah dikonfirmasi
  komunitas Hugging Face:
    - https://huggingface.co/vidore/colpali-v1.3-hf/discussions/6
    - https://github.com/huggingface/transformers/issues/38927
  (Catatan: ini PERSIS pola masalah yang sama dengan ColQwen2 sebelumnya —
  bukan kebetulan, sumbernya sama: colpali_engine vs transformers terbaru.)

SOLUSI: Pakai checkpoint **vidore/colpali-v1.3-hf** — versi LoRA-nya SUDAH
DIGABUNG (merged) ke bobot dasar, dan dimuat lewat class native
`transformers.ColPaliForRetrieval` (BUKAN colpali_engine sama sekali). Tidak
ada adapter terpisah -> tidak ada lagi masalah penamaan key. Ini persis pola
yang sama dengan dokumen resmi ColQwen2 yang Anda kirim di awal percakapan
kita (model_doc/colqwen2), hanya untuk ColPali (model_doc/colpali).

DAFTAR PERUBAHAN dari script sebelumnya:
1. DIHAPUS total: `colpali_engine` (sumber semua bug LoRA/adapter selama ini).
2. Model    : ColPali("vidore/colpali-v1.3") via colpali_engine
              -> ColPaliForRetrieval("vidore/colpali-v1.3-hf") via transformers
3. Processor: ColPaliProcessor (colpali_engine), method process_images()
              -> ColPaliProcessor (transformers), method __call__ biasa:
              processor(images=[...], return_tensors="pt")
4. Output model: dulu model(**batch) -> tensor langsung
              -> sekarang model(**batch).embeddings (ada atribut .embeddings)
5. Collection BARU lagi: "colpali_v13_hf_multivec_rag_index" — index lama
   "colpali_v13_multivec_rag_index" (hasil colpali_engine) HARUS dianggap
   rusak juga (LoRA-nya random-init), jangan dipakai.
6. torchao TIDAK PERLU di-uninstall lagi — karena tidak ada peft/LoRA yang
   dimuat sama sekali di pipeline ini sekarang.
7. TETAP TIDAK ada normalisasi L2 manual sebelum upsert ke Qdrant — ini
   konsisten dengan CONTOH RESMI dari dokumentasi transformers utk
   ColPaliForRetrieval & ColQwen2ForRetrieval (keduanya tidak menormalisasi
   sebelum scoring). Distance.DOT tetap dipakai di Qdrant (bukan COSINE).
8. PyMuPDF (fitz) tetap dipakai utk ekstraksi PDF->PNG (bukan pdf2image).
9. GPU 4-bit (bitsandbytes) + checkpoint-resume + diagnostik memori tetap
   dipertahankan (poin optimisasi RAM/VRAM).
=============================================================================
"""

# %% [Cell 0] Install dependency — colpali_engine & torchao-uninstall DIHAPUS
# (tidak diperlukan lagi sama sekali, lihat poin #1 & #6 di atas)
!pip install -q --no-cache-dir \
    "transformers>=4.49.0" \
    qdrant-client \
    bitsandbytes \
    accelerate \
    pymupdf \
    pillow \
    tqdm \
    psutil


In [ ]:
# [Cell 1] Imports & Konfigurasi
import os
import gc
import re
import time
import json
import hashlib
import pathlib
import traceback
import torch
import psutil
import fitz  # PyMuPDF
from PIL import Image
from tqdm import tqdm

from qdrant_client import QdrantClient
from qdrant_client import models

# >>> Native transformers, BUKAN colpali_engine (lihat docstring poin #1-#4)
from transformers import ColPaliForRetrieval, ColPaliProcessor, BitsAndBytesConfig
from google.colab import drive, userdata

# =========================================================
# 1. KONFIGURASI
# =========================================================
RETRIEVAL_MODEL_ID = "vidore/colpali-v1.3-hf"   # <-- checkpoint MERGED, bukan LoRA
VECTOR_DIM          = 128
COLLECTION_NAME      = "colpali_v13_hf_multivec_rag_index"   # <-- nama BARU lagi (poin #5)

PDF_SOURCE_FOLDER        = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/Dataset"
IMAGE_DESTINATION_FOLDER = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/Dataset/extracted_images"
CHECKPOINT_FILE          = "/content/drive/MyDrive/Thesis-Agung_Prawoto-56225110019/Dataset/extracted_images/indexing_checkpoint_colpali_v13_hf.json"

EXTRACTION_DPI = 200
MAX_IMAGE_SIZE = 2048

# =========================================================
# 2. UTILITAS MEMORI
# =========================================================
def clean_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def print_mem_usage(label: str = ""):
    tag = f" [{label}]" if label else ""
    ram = psutil.virtual_memory()
    print(f"  [MEM]{tag} RAM : {ram.used/1024**3:.2f} / {ram.total/1024**3:.2f} GB ({ram.percent:.0f}%)")
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  [MEM]{tag} VRAM: alloc={alloc:.2f}GB reserved={reserved:.2f}GB")

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {}

def save_checkpoint(checkpoint):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)

def extract_pdf_name(filename: str) -> str:
    match = re.match(r"^(.*?)_p(\d{4})\.png$", filename, re.IGNORECASE)
    if match:
        return match.group(1)
    base = os.path.splitext(filename)[0]
    parts = base.split("_")
    if len(parts) > 1:
        return "_".join(parts[:-1])
    return base

In [ ]:

# =========================================================
# 3. EKSTRAKSI PDF -> PNG (PyMuPDF, tidak berubah)
# =========================================================
def extract_images_from_pdfs(pdf_input_folder: str, output_image_folder: str, dpi: int = EXTRACTION_DPI):
    pathlib.Path(output_image_folder).mkdir(parents=True, exist_ok=True)
    pdf_files = sorted([f for f in os.listdir(pdf_input_folder) if f.lower().endswith(".pdf")])

    if not pdf_files:
        print(f"  [WARN] Tidak ada PDF ditemukan di: {pdf_input_folder}")
        return

    print(f"  Ditemukan {len(pdf_files)} PDF | DPI={dpi}")
    mat = fitz.Matrix(dpi / 72.0, dpi / 72.0)

    for idx, pdf_name in enumerate(pdf_files, 1):
        pdf_base = os.path.splitext(pdf_name)[0]

        if any(f.startswith(pdf_base + "_p") for f in os.listdir(output_image_folder)):
            print(f"  [SKIP] {pdf_name} — sudah diekstrak sebelumnya.")
            continue

        pdf_path = os.path.join(pdf_input_folder, pdf_name)
        try:
            with fitz.open(pdf_path) as doc:
                n = len(doc)
                for page_num in range(n):
                    out_path = os.path.join(output_image_folder, f"{pdf_base}_p{page_num:04d}.png")
                    pix = doc.load_page(page_num).get_pixmap(matrix=mat, colorspace=fitz.csRGB)
                    pix.save(out_path)
                    pix = None
            print(f"  [{idx}/{len(pdf_files)}] OK {pdf_name} ({n} halaman diekstrak)")
        except Exception as e:
            print(f"  [{idx}/{len(pdf_files)}] GAGAL {pdf_name}: {e}")
        clean_mem()

    total_pngs = len([f for f in os.listdir(output_image_folder) if f.endswith(".png")])
    print(f"\n  Total PNG di folder tujuan: {total_pngs}\n")

# =========================================================
# 4. QDRANT: BUAT COLLECTION MULTIVECTOR (MaxSim, Distance.DOT)
# =========================================================
def ensure_collection(client):
    try:
        info = client.get_collection(COLLECTION_NAME)
        print(f"  [INFO] Collection '{COLLECTION_NAME}' sudah ada ({info.points_count} points).")
    except Exception:
        print(f"  [INFO] Membuat collection baru '{COLLECTION_NAME}'...")
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=models.VectorParams(
                size=VECTOR_DIM,
                distance=models.Distance.DOT,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                ),
            ),
        )
        print(f"  [OK] Collection dibuat (dim={VECTOR_DIM}, distance=DOT, comparator=MAX_SIM).")

# =========================================================
# 5. PIPELINE EMBEDDING (ColPaliForRetrieval native — poin #2,#3,#4)
# =========================================================
def embed_single_image(img_path, model, processor, device):
    Image.MAX_IMAGE_PIXELS = None
    img = Image.open(img_path).convert("RGB")

    if max(img.size) > MAX_IMAGE_SIZE:
        img.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), Image.Resampling.LANCZOS)

    orig_size = img.size
    # API native transformers: processor(images=[...]) bukan process_images([...])
    inputs = processor(images=[img], return_tensors="pt").to(device)
    img.close()
    del img

    # PAKAI torch.no_grad(), BUKAN torch.inference_mode() — inference_mode()
    # terlalu ketat utk beberapa operasi buffer/cache dinamis di internal
    # PaliGemma/SigLIP (mis. position embedding utk resolusi gambar yang
    # berubah-ubah antar halaman PDF), menyebabkan error:
    # "data set to a tensor that requires gradients must be floating point
    # or complex dtype". no_grad() tetap menonaktifkan gradient (cukup utk
    # inference) tapi tidak punya batasan ekstra itu.
    with torch.no_grad():
        output = model(**inputs)
        # Output punya atribut .embeddings (beda dari colpali_engine yg
        # mengembalikan tensor langsung) — TANPA normalisasi manual, sesuai
        # contoh resmi dokumentasi transformers.
        vec = output.embeddings[0].float().cpu().numpy()

    del inputs, output
    return vec, orig_size

def run_indexing():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n  Device : {device}")
    if device == "cuda":
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  VRAM   : {vram:.1f} GB")
    print_mem_usage("awal run_indexing")

    checkpoint = load_checkpoint()

    HF_TOKEN       = userdata.get("HF_TOKEN")
    qdrant_url     = userdata.get("QDRANT_URL")
    qdrant_api_key = userdata.get("QDRANT_API_KEY")

    print(f"\nMemuat model: {RETRIEVAL_MODEL_ID} (bfloat16, GPU, native transformers)...")
    processor = ColPaliProcessor.from_pretrained(RETRIEVAL_MODEL_ID, token=HF_TOKEN)
    # >>> PERUBAHAN: TIDAK pakai 4-bit (bitsandbytes) lagi di sini.
    # Error "data set to a tensor that requires gradients must be floating
    # point or complex dtype" konsisten muncul di SETIAP gambar (bukan
    # tergantung data) -> indikasi kuat ini bug interaksi bitsandbytes 4-bit
    # dgn operasi buffer/cache dinamis di model (4-bit weight bitsandbytes
    # disimpan sbg buffer uint8 internal, rawan konflik dgn operasi semacam
    # ini). Karena Stage 1 cuma memuat SATU model (ColPali) sendirian —
    # tidak ada LLM lain bersamaan — VRAM T4 (15GB) masih sangat cukup utk
    # ColPali di bfloat16 penuh (~6GB), jadi 4-bit tidak diperlukan di sini.
    model = ColPaliForRetrieval.from_pretrained(
        RETRIEVAL_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="cuda:0" if device == "cuda" else "cpu",
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
    ).eval()
    print_mem_usage("setelah load model ColPali (native)")
    # Tidak ada LOAD REPORT dgn MISSING/UNEXPECTED lagi yg diharapkan di sini,
    # karena tidak ada adapter LoRA terpisah. Kalau MASIH muncul LOAD REPORT
    # dgn banyak MISSING, itu tanda ada masalah lain -> kirim log ke saya.

    client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
    ensure_collection(client)

    all_images = sorted([f for f in os.listdir(IMAGE_DESTINATION_FOLDER) if f.lower().endswith(".png")])
    pending = [f for f in all_images if f not in checkpoint]

    print(f"\n  Total gambar diekstrak : {len(all_images)}")
    print(f"  Sudah diindeks         : {len(all_images) - len(pending)}")
    print(f"  Pending indexing       : {len(pending)}\n")

    if not pending:
        print("  Semua gambar sudah diindeks.")
        return

    t_start = time.time()
    n_failed = 0
    traceback_shown = False  # supaya tidak banjir log, hanya print full traceback 1x

    for i, img_name in enumerate(tqdm(pending, desc="Indexing ColPali(hf)->Qdrant")):
        img_path = os.path.join(IMAGE_DESTINATION_FOLDER, img_name)

        try:
            vec, size = embed_single_image(img_path, model, processor, device)
        except torch.cuda.OutOfMemoryError:
            print(f"  [OOM] {img_name} — skip gambar ini, bersihkan memori.")
            clean_mem()
            n_failed += 1
            continue
        except Exception as e:
            print(f"  [ERR] Embedding gagal utk {img_name}: {e}")
            if not traceback_shown:
                print("  [DEBUG] Traceback lengkap (hanya ditampilkan 1x utk error pertama):")
                print(traceback.format_exc())
                traceback_shown = True
            n_failed += 1
            continue

        pdf_name = extract_pdf_name(img_name)

        point = models.PointStruct(
            id=int(hashlib.md5(img_name.encode()).hexdigest(), 16) % (10**15),
            vector=vec.tolist(),
            payload={
                "file": img_name,
                "pdf_name": pdf_name,
                "input_size": list(size),
                "num_patches": vec.shape[0],
                "model": "ColPali-v1.3-hf",
            },
        )
        del vec

        try:
            client.upsert(collection_name=COLLECTION_NAME, points=[point])
        except Exception as e:
            print(f"  [UPSERT ERR] {img_name}: {e}")
            n_failed += 1
            del point
            continue
        del point

        checkpoint[img_name] = True
        if (i + 1) % 10 == 0:
            save_checkpoint(checkpoint)

        clean_mem()
        if (i + 1) % 25 == 0:
            print_mem_usage(f"setelah {i+1} gambar")

    save_checkpoint(checkpoint)
    print_mem_usage("akhir run_indexing")
    print("\n" + "=" * 50)
    print("STAGE 1 INDEXING (ColPali v1.3-hf -> Qdrant) SELESAI")
    print("=" * 50)
    print(f"  Model ID   : {RETRIEVAL_MODEL_ID}")
    print(f"  Collection : {COLLECTION_NAME}")
    print(f"  Berhasil   : {len(pending) - n_failed}")
    print(f"  Gagal      : {n_failed}")
    print("=" * 50 + "\n")

# =========================================================
# 6. MAIN
# =========================================================
def main():
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass

    print("\n" + "=" * 55)
    print(f"STAGE 1a: Ekstraksi PDF -> PNG (DPI={EXTRACTION_DPI})")
    print("=" * 55)
    extract_images_from_pdfs(PDF_SOURCE_FOLDER, IMAGE_DESTINATION_FOLDER)

    print("\n" + "=" * 55)
    print("STAGE 1b: Indexing Multi-Vector ColPali v1.3-hf -> Qdrant")
    print("=" * 55)
    run_indexing()

if __name__ == "__main__":
    main()